# RunPod GPU 모델 비교 측정

**이 노트북 파일 하나만 있으면 됩니다.** 위에서부터 한 칸씩 `Shift + Enter` 로 실행하세요.
필요한 모델·사진·코드(약 700MB)는 첫 칸이 구글 드라이브에서 받아 검증하고 풀어 줍니다. 보낸 사람이 알려 준 **구글 드라이브 링크**를 첫 칸에서 한 번 붙여넣게 됩니다.

1. 첫 칸: 구글 드라이브에서 파일 받기(검증·압축 풀기 포함), 작업 폴더와 GPU 확인
2. 둘째 칸: 파이썬 패키지 확인(없으면 자동 설치)
3. 셋째 칸: 모델 파일 확인
4. 넷째 칸: 사진 폴더 확인
5. 다섯째 칸: 측정 실행(몇 분 걸립니다)
6. 여섯째 칸: 결과를 zip 하나로 묶기 → 파일 목록에서 내려받아 보내 주세요

끝나면 **Pod 를 반드시 정지(Stop)하거나 삭제(Terminate)** 하세요. 켜 두면 계속 과금됩니다.

In [ ]:
import hashlib
import os
import subprocess
import sys
import zipfile
from pathlib import Path

# 보낸 사람이 알려 준 구글 드라이브 공유 링크. 그대로 두면 실행할 때 붙여넣으라고 물어본다
DRIVE_URL = "여기에_구글드라이브_공유링크"
ZIP_BYTES = 695774050
ZIP_SHA256 = "7b2f04d559427de9ed30afcc70b775c6706b8a5f8e541b8b7ab01d0174ab2bb3"

BASE = Path("/workspace") if Path("/workspace").is_dir() else Path.cwd()
ROOT = BASE / "pair-bench"
ZIP_PATH = BASE / "runpod-all-in-one.zip"


def sha256_of(path):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


if (ROOT / "src" / "pair_vision").is_dir():
    print("이미 받아 둔 파일이 있어 다운로드를 건너뜁니다:", ROOT)
else:
    if not ZIP_PATH.is_file():  # 압축 파일을 직접 올려 두었으면 받지 않는다
        url = DRIVE_URL if DRIVE_URL.startswith("http") else input("구글 드라이브 공유 링크를 붙여넣고 Enter: ").strip()
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown

        print("받는 중 (약 700MB, 몇 분 걸릴 수 있습니다)...")
        gdown.download(url, str(ZIP_PATH), quiet=False, fuzzy=True)
    if not ZIP_PATH.is_file():
        raise RuntimeError("파일을 받지 못했습니다. 링크가 '링크가 있는 모든 사용자'로 공유돼 있는지 확인하세요.")
    print("검증 중...")
    if ZIP_PATH.stat().st_size != ZIP_BYTES or sha256_of(ZIP_PATH) != ZIP_SHA256:
        raise RuntimeError(f"받은 파일이 보낸 파일과 다릅니다(용량·해시 불일치). {ZIP_PATH} 를 지우고 다시 받으세요.")
    print("압축 푸는 중...")
    with zipfile.ZipFile(ZIP_PATH) as archive:
        archive.extractall(ROOT)
    ZIP_PATH.unlink()
    print("준비 완료:", ROOT)

os.chdir(ROOT)
sys.path[:0] = [str(ROOT / "src"), str(ROOT)]

# 필요하면 바꾸는 값 (환경변수로도 줄 수 있다)
MODELS = os.environ.get("MODELS", "mediapipe-hands,dfine-s,rtdetr-r18,owlv2")
WARMUP = int(os.environ.get("WARMUP", "5"))
REPEAT = int(os.environ.get("REPEAT", "10"))
ALLOW_CPU = os.environ.get("ALLOW_CPU") == "1"  # 내 PC 에서 시험할 때만 쓴다. Pod 에서는 쓰지 않는다

print("작업 폴더:", ROOT)
try:
    gpu_line = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
        capture_output=True, text=True, check=True,
    ).stdout.strip()
    print("GPU:", gpu_line)
except (FileNotFoundError, subprocess.CalledProcessError):
    print("nvidia-smi 를 쓸 수 없습니다 (GPU 가 없는 환경)")


In [ ]:
def ensure(import_name, pip_name=None):
    """패키지가 없으면 pip 로 설치한다. 성공 여부를 돌려준다."""
    try:
        __import__(import_name)
        return True
    except ImportError:
        pass
    result = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name or import_name])
    return result.returncode == 0


for import_name, pip_name in [("transformers", None), ("scipy", None), ("PIL", "pillow"), ("numpy", None), ("mediapipe", None)]:
    print(f"{import_name:14s}", "OK" if ensure(import_name, pip_name) else "설치 실패 (이 패키지가 필요한 모델은 건너뜁니다)")

import torch

print("torch", torch.__version__, "| cuda 사용 가능:", torch.cuda.is_available())
if not torch.cuda.is_available() and not ALLOW_CPU:
    raise RuntimeError("CUDA 를 쓸 수 없습니다. PyTorch+CUDA 템플릿의 GPU Pod 에서 실행하세요 (이 상태로 재면 CPU 값이 됩니다).")


In [ ]:
NEEDED = {
    "rtdetr_r18vd_coco_o365": "PekingU/rtdetr_r18vd_coco_o365",
    "dfine-small-coco": "ustc-community/dfine-small-coco",
    "owlv2-base-patch16-ensemble": "google/owlv2-base-patch16-ensemble",
}
for folder, repo in NEEDED.items():
    target = ROOT / "models" / folder
    if (target / "config.json").is_file():
        print("있음  :", folder)
        continue
    # 압축 파일에 있어야 하지만 빠져 있으면 공개 저장소에서 받는다
    print("받는 중:", repo)
    from huggingface_hub import snapshot_download

    snapshot_download(repo, local_dir=target, ignore_patterns=["*.bin", "*.onnx", "*.msgpack", "*.h5", "*.ot"])
hand = ROOT / "models" / "hand_landmarker.task"
print("손 검출 모델:", "있음" if hand.is_file() else "없음 (손 검출은 건너뜁니다)")


In [ ]:
IMAGE_SUFFIXES = {".png", ".jpg", ".jpeg"}


def has_images(folder):
    return folder.is_dir() and any(p.suffix.lower() in IMAGE_SUFFIXES for p in folder.iterdir())


PHOTOS = next((c for c in (ROOT / "photos", Path("/workspace/photos")) if has_images(c)), None)
if PHOTOS is None:
    # 사진이 없으면 속도 측정용 합성 사진을 만든다(검출 결과는 의미 없고 속도만 본다)
    import numpy as np
    from PIL import Image, ImageDraw

    PHOTOS = Path("/tmp/synthetic-photos")
    PHOTOS.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(0)
    for i in range(8):
        image = Image.fromarray((rng.random((380, 480, 3)) * 60 + 120).astype("uint8"))
        draw = ImageDraw.Draw(image)
        for _ in range(6):
            x0, y0 = int(rng.integers(0, 380)), int(rng.integers(0, 280))
            w, h = int(rng.integers(40, 100)), int(rng.integers(30, 80))
            draw.rectangle([x0, y0, x0 + w, y0 + h], fill=tuple(int(v) for v in rng.integers(30, 220, 3)))
        image.save(PHOTOS / f"{i + 1}.png")
    print("사진 폴더가 없어 합성 사진 8장을 만들었습니다:", PHOTOS)
else:
    print("사진 폴더:", PHOTOS, "|", sum(1 for p in PHOTOS.iterdir() if p.suffix.lower() in IMAGE_SUFFIXES), "장")


In [ ]:
import contextlib
import io
import re

from tools.bench import compare

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
SAFE = re.sub(r"[^A-Za-z0-9._-]+", "_", gpu_name)
OUT = ROOT / "results"
OUT.mkdir(exist_ok=True)

buffer = io.StringIO()
with contextlib.redirect_stdout(buffer):
    exit_code = compare.main([
        "--images", str(PHOTOS), "--models", MODELS,
        "--warmup", str(WARMUP), "--repeat", str(REPEAT),
        "--json", str(OUT / f"result_{SAFE}.json"),
    ])
report = buffer.getvalue()
print(report)
(OUT / f"result_{SAFE}.md").write_text(report, encoding="utf-8")
print("종료 코드:", exit_code, "(0 이 정상)")
print("참고: mediapipe-hands(손 검출)는 GPU 가 아니라 이 Pod 의 CPU 로 돕니다.")


## (선택) 용량 측정: 응시자 몇 명까지 감당하나
가상 응시자 N명이 웹캠·휴대폰 사진을 초당 1장씩 보낼 때 응시자당 0.9fps 이상을 유지하는 최대 인원을, 워커 수·GPU 묶음·fp16·웹캠 서버 분석 여부 조합별로 잽니다.
조합이 많아 **전부 돌리면 1~2시간** 걸립니다. 먼저 `QUICK = True` 로 한 조합만 확인한 뒤 `False` 로 바꿔 전체를 돌리세요. 결과는 `results/capacity_*.md` / `.csv` 로 남고, 아래 zip 칸이 같이 묶어 줍니다.

In [ ]:
QUICK = True  # True: 한 조합(워커 2, 묶음 on, fp16 on, 웹캠 server, 10·20명)만. False: 전체 조합

args = ["--detector", "real", "--no-verify", "--photos", str(PHOTOS), "--models-dir", str(ROOT / "models"),
        "--out", str(OUT / f"capacity_{SAFE}")]
if QUICK:
    args += ["--workers", "2", "--batch", "on", "--fp16", "on", "--webcam", "server", "--examinees", "10,20", "--duration", "15"]
# 일부만 고르려면 위 인자를 바꾼다: --workers 1,2,4 --batch off,on --fp16 off,on --webcam server,off --examinees 10,20,30,40
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nvidia-ml-py", "psutil"], check=False)
env = {**os.environ, "PYTHONPATH": f"{ROOT / 'src'}{os.pathsep}{ROOT}"}
subprocess.run([sys.executable, "-u", "-m", "tools.bench.capacity", *args], cwd=ROOT, env=env, check=False)
print((OUT / f"capacity_{SAFE}.md").read_text(encoding="utf-8")[:3000])

In [ ]:
import importlib.metadata as metadata
import platform
import zipfile


def version(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return "없음"


lines = [
    f"GPU: {gpu_name}",
    f"python: {platform.python_version()} | {platform.platform()}",
    *(f"{n}: {version(n)}" for n in ("torch", "transformers", "mediapipe", "numpy", "pillow")),
]
try:
    lines += ["", subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=True).stdout]
except (FileNotFoundError, subprocess.CalledProcessError):
    pass
(OUT / "env.txt").write_text("\n".join(lines), encoding="utf-8")

zip_path = OUT / f"results_{SAFE}.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for name in (f"result_{SAFE}.md", f"result_{SAFE}.json", "env.txt"):
        archive.write(OUT / name, name)
print("만들었습니다:", zip_path)
print("왼쪽 파일 목록에서 results 폴더 안의 이 zip 을 우클릭 -> Download 로 내려받아 보내 주세요.")
try:
    from IPython.display import FileLink, display

    display(FileLink(str(zip_path.relative_to(ROOT))))
except ImportError:
    pass


**끝났습니다. 이제 Pod 를 정지(Stop)하거나 삭제(Terminate)하세요.** 켜 두면 계속 과금됩니다.